In [12]:
# ============================================================
# 0. Импорты и настройки
# ============================================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 20          # базовый шрифт (оси, тики, текст)
plt.rcParams['axes.titlesize'] = 20     # заголовки осей
plt.rcParams['axes.labelsize'] = 22     # подписи осей X/Y
plt.rcParams['xtick.labelsize'] = 18    # цифры на осях
plt.rcParams['ytick.labelsize'] = 18
plt.rcParams['legend.fontsize'] = 16    # легенда
plt.rcParams['figure.titlesize'] = 20
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:bold'
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 6
plt.rcParams['ytick.major.size'] = 6

OUT_DIR = 'results'
os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# 1. Загрузка и очистка
# ============================================================
FILE = 'database-2-2026-10-02.xlsx'
df = pd.read_excel(FILE, sheet_name='Лист1')

num_cols = ['D_n', 'bar_e_0', 'p_nom', 'delta_I_ud', 'delta_I_p', 'delta_t_p']
for col in num_cols:
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(',', '.', regex=False).str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

df_all = df.dropna(subset=['delta_I_ud', 'delta_I_p', 'delta_t_p']).reset_index(drop=True)
print("Всего строк:", len(df), "| валидных:", len(df_all))

TARGETS = ['delta_I_p', 'delta_t_p', 'delta_I_ud']
X_COL      = 'D_n'
GROUP_PNOM = 'p_nom'
GROUP_BAR  = 'bar_e_0'

P_VALS  = sorted(df_all[GROUP_PNOM].unique())
BE_VALS = sorted(df_all[GROUP_BAR].unique())

LABELS = {
    'delta_I_p':  r'$\delta_{I_п}$, %',
    'delta_I_ud': r'$\delta_{I_{уд}}$, %',
    'delta_t_p':  r'$\delta_{t_п}$, %',
}
SYMBOL = {
    'delta_I_p':  r'\delta_{I_п}',
    'delta_I_ud': r'\delta_{I_{уд}}',
    'delta_t_p':  r'\delta_{t_п}',
}

FN_LABEL = 'функциональная зависимость'


# ============================================================
# 2. Метрики
# ============================================================
def sko_sao(y_true, y_pred):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    err = y_true - y_pred
    return (float(np.sqrt(np.mean(err**2))),
            float(np.mean(np.abs(err))),
            float(r2_score(y_true, y_pred)))

def aic_bic(y, y_pred, k):
    n = len(y)
    rss = float(np.sum((np.asarray(y, float) - np.asarray(y_pred, float))**2))
    rss = max(rss, 1e-300)
    return (n * np.log(rss / n) + 2 * k,
            n * np.log(rss / n) + k * np.log(n))


# ============================================================
# 3. БАЗОВЫЙ СТЕПЕННОЙ ЗАКОН (для линий МНК)
# ============================================================
def fit_power_law(df, target):
    X = df[['D_n', 'p_nom', 'bar_e_0']].values.astype(float)
    y = df[target].values.astype(float)
    m = LinearRegression().fit(np.log(X), np.log(y))
    C = float(np.exp(m.intercept_))
    a, b, c = m.coef_
    return C, a, b, c

def power_law(C, a, b, c, D, p, be):
    return C * (D ** a) * (p ** b) * (be ** c)


# ============================================================
# 4. ГИБРИДНЫЕ МОДЕЛИ
# ============================================================
def model_H1(X, C, a, b, c):
    D, p, be = X
    return C * D**a * p**b * be**c

def model_H2(X, C, a, b, c, k):
    D, p, be = X
    return C * D**a * p**b * be**c * np.exp(k * D)

def model_H3(X, C, a, b, c, m):
    D, p, be = X
    return C * D**a * p**b * be**c * np.exp(m * be)

def model_H4(X, C, a, b, c, k, m):
    D, p, be = X
    return C * D**a * p**b * be**c * np.exp(k * D + m * be)

def model_H5(X, C, a, b, c, k, m, n):
    D, p, be = X
    return C * D**a * p**b * be**c * np.exp(k * D + m * be + n * p)

def model_H6(X, C, a, b, c, A, k):
    D, p, be = X
    return C * D**a * p**b * be**c + A * np.exp(k * D)

def model_H7(X, C1, a1, b1, c1, C2, a2, b2, c2):
    D, p, be = X
    return (C1 * D**a1 * p**b1 * be**c1
            + C2 * D**a2 * p**b2 * be**c2)

def model_H8(X, C, a, q, b, c):
    D, p, be = X
    return C * D**(a + q * be) * p**b * be**c

def model_H9(X, C, a, r, b, c):
    D, p, be = X
    return C * D**(a + r * p) * p**b * be**c


# ============================================================
# 5. ФОРМАТИРОВАНИЕ (возвращают строку С $...$)
# ============================================================
def _num(x, nd=3):
    if x == 0:
        return "0"
    if abs(x) < 1e-3 or abs(x) >= 1e4:
        mant, exp = f"{x:.2e}".split("e")
        mant = mant.rstrip("0").rstrip(".")
        return rf"{mant}\cdot10^{{{int(exp)}}}"
    return f"{x:.{nd}f}".rstrip("0").rstrip(".")

def format_model(name, params, target):
    """Только для формулы функциональной зависимости (в заголовке)."""
    P = SYMBOL[target]
    tag = name.split(":")[0].strip()

    if tag == "H1":
        C, a, b, c = params
        s = (rf"{P} = {_num(C)}\cdot D_n^{{{_num(a)}}}"
             rf"\cdot p_{{ном}}^{{{_num(b)}}}\cdot\bar{{e}}_0^{{{_num(c)}}}")
    elif tag == "H2":
        C, a, b, c, k = params
        s = (rf"{P} = {_num(C)}\cdot D_n^{{{_num(a)}}}"
             rf"\cdot p_{{ном}}^{{{_num(b)}}}\cdot\bar{{e}}_0^{{{_num(c)}}}"
             rf"\cdot e^{{{_num(k)}D_n}}")
    elif tag == "H3":
        C, a, b, c, m = params
        s = (rf"{P} = {_num(C)}\cdot D_n^{{{_num(a)}}}"
             rf"\cdot p_{{ном}}^{{{_num(b)}}}\cdot\bar{{e}}_0^{{{_num(c)}}}"
             rf"\cdot e^{{{_num(m)}\bar{{e}}_0}}")
    elif tag == "H4":
        C, a, b, c, k, m = params
        s = (rf"{P} = {_num(C)}\cdot D_n^{{{_num(a)}}}"
             rf"\cdot p_{{ном}}^{{{_num(b)}}}\cdot\bar{{e}}_0^{{{_num(c)}}}"
             rf"\cdot e^{{{_num(k)}D_n + {_num(m)}\bar{{e}}_0}}")
    elif tag == "H5":
        C, a, b, c, k, m, n = params
        s = (rf"{P} = {_num(C)}\cdot D_n^{{{_num(a)}}}"
             rf"\cdot p_{{ном}}^{{{_num(b)}}}\cdot\bar{{e}}_0^{{{_num(c)}}}"
             rf"\cdot e^{{{_num(k)}D_n + {_num(m)}\bar{{e}}_0 + {_num(n)}p_{{ном}}}}")
    elif tag == "H6":
        C, a, b, c, A, k = params
        s = (rf"{P} = {_num(C)}\cdot D_n^{{{_num(a)}}}"
             rf"\cdot p_{{ном}}^{{{_num(b)}}}\cdot\bar{{e}}_0^{{{_num(c)}}}"
             rf" + {_num(A)}\cdot e^{{{_num(k)}D_n}}")
    elif tag == "H7":
        C1, a1, b1, c1, C2, a2, b2, c2 = params
        s = (rf"{P} = {_num(C1)}\cdot D_n^{{{_num(a1)}}}"
             rf"\cdot p_{{ном}}^{{{_num(b1)}}}\cdot\bar{{e}}_0^{{{_num(c1)}}}"
             rf" + {_num(C2)}\cdot D_n^{{{_num(a2)}}}"
             rf"\cdot p_{{ном}}^{{{_num(b2)}}}\cdot\bar{{e}}_0^{{{_num(c2)}}}")
    elif tag == "H8":
        C, a, q, b, c = params
        s = (rf"{P} = {_num(C)}\cdot D_n^{{{_num(a)} + {_num(q)}\bar{{e}}_0}}"
             rf"\cdot p_{{ном}}^{{{_num(b)}}}\cdot\bar{{e}}_0^{{{_num(c)}}}")
    elif tag == "H9":
        C, a, r, b, c = params
        s = (rf"{P} = {_num(C)}\cdot D_n^{{{_num(a)} + {_num(r)}p_{{ном}}}}"
             rf"\cdot p_{{ном}}^{{{_num(b)}}}\cdot\bar{{e}}_0^{{{_num(c)}}}")
    else:
        s = rf"{P} = \ldots"

    return f'${s}$'


# ============================================================
# 6. Подгонка
# ============================================================
def fit_model(df, target, model, p0, bounds, name):
    D  = df['D_n'].values.astype(float)
    p  = df['p_nom'].values.astype(float)
    be = df['bar_e_0'].values.astype(float)
    y  = df[target].values.astype(float)

    popt, _ = curve_fit(model, (D, p, be), y,
                        p0=p0, maxfev=300000, bounds=bounds)

    def predict(d, pp, bbe):
        return model((np.asarray(d, float),
                      np.asarray(pp, float),
                      np.asarray(bbe, float)), *popt)

    y_pred = predict(D, p, be)
    rmse, mae, r2 = sko_sao(y, y_pred)
    aic, bic = aic_bic(y, y_pred, k=len(popt))

    return popt, predict, dict(
        name=name, params=popt,
        R2=r2, SAO=mae, SKO=rmse,
        AIC=aic, BIC=bic, k=len(popt),
        formula=format_model(name, popt, target)
    )


# ============================================================
# 7. Сравнение моделей (по AIC)
# ============================================================
def compare_all_models(df, target):
    C0, a0, b0, c0 = fit_power_law(df, target)

    B_C, B_a, B_b, B_c = (1e-8, 1e+3), (-5.0, 0.0), (-3.0, 3.0), (-5.0, 5.0)
    B_k, B_m, B_n = (-20.0, 20.0), (-20.0, 20.0), (-1e-5, 1e-5)
    B_A = (1e-6, 1e+3)

    specs = [
        ('H1: степень (МНК-ядро)', model_H1, [C0, a0, b0, c0],
         ([B_C[0], B_a[0], B_b[0], B_c[0]], [B_C[1], B_a[1], B_b[1], B_c[1]])),
        ('H2: степень·exp(k·D)', model_H2, [C0, a0, b0, c0, -1.0],
         ([B_C[0], B_a[0], B_b[0], B_c[0], B_k[0]], [B_C[1], B_a[1], B_b[1], B_c[1], B_k[1]])),
        ('H3: степень·exp(m·ē)', model_H3, [C0, a0, b0, c0, -1.0],
         ([B_C[0], B_a[0], B_b[0], B_c[0], B_m[0]], [B_C[1], B_a[1], B_b[1], B_c[1], B_m[1]])),
        ('H4: степень·exp(k·D+m·ē)', model_H4, [C0, a0, b0, c0, -1.0, -1.0],
         ([B_C[0], B_a[0], B_b[0], B_c[0], B_k[0], B_m[0]], [B_C[1], B_a[1], B_b[1], B_c[1], B_k[1], B_m[1]])),
        ('H5: степень·exp(k·D+m·ē+n·p)', model_H5, [C0, a0, b0, c0, -1.0, -1.0, 0.0],
         ([B_C[0], B_a[0], B_b[0], B_c[0], B_k[0], B_m[0], B_n[0]], [B_C[1], B_a[1], B_b[1], B_c[1], B_k[1], B_m[1], B_n[1]])),
        ('H6: степень + A·exp(k·D)', model_H6, [C0, a0, b0, c0, 0.1, -1.0],
         ([B_C[0], B_a[0], B_b[0], B_c[0], B_A[0], B_k[0]], [B_C[1], B_a[1], B_b[1], B_c[1], B_A[1], B_k[1]])),
        ('H7: сумма двух степеней', model_H7,
         [max(C0/2, 1e-3), a0, b0, c0, max(C0/2, 1e-3), a0*0.5, b0, c0],
         ([1e-4, -5.0, -3.0, -5.0, 1e-4, -5.0, -3.0, -5.0],
          [1e+1, 0.0, 3.0, 5.0, 1e+1, 0.0, 3.0, 5.0])),
        ('H8: D^(a+q·ē)·p^b·ē^c', model_H8, [C0, a0, 0.0, b0, c0],
         ([B_C[0], B_a[0], -10.0, B_b[0], B_c[0]], [B_C[1], B_a[1], 10.0, B_b[1], B_c[1]])),
        ('H9: D^(a+r·p)·p^b·ē^c', model_H9, [C0, a0, 0.0, b0, c0],
         ([B_C[0], B_a[0], -1e-5, B_b[0], B_c[0]], [B_C[1], B_a[1], 1e-5, B_b[1], B_c[1]])),
    ]

    rows, candidates = [], []
    for name, model, p0, bounds in specs:
        try:
            popt, predict, met = fit_model(df, target, model, p0, bounds, name)
            rows.append((name, met['R2'], met['SAO'], met['SKO'], met['AIC'], met['k']))
            candidates.append((met['AIC'], name, predict, met))
        except Exception as e:
            rows.append((name, np.nan, np.nan, np.nan, np.nan, np.nan))
            print(f'[!] {name}: {e}')

    res = pd.DataFrame(rows, columns=['Модель', 'R²', 'САО (MAE)',
                                      'СКО (RMSE)', 'AIC', 'k']).sort_values('AIC').reset_index(drop=True)

    print("\n" + "="*90)
    print(f"СРАВНЕНИЕ МОДЕЛЕЙ: {target}  (сортировка по AIC)")
    print("="*90)
    print(res.round(6).to_string(index=False))

    best = min(candidates, key=lambda t: t[0])
    _, best_name, best_predict, best_met = best
    print(f"\n>>> ЛУЧШАЯ ПО AIC: {best_name}")
    print(f"    R²={best_met['R2']:.6f}, СКО={best_met['SKO']:.6f}, AIC={best_met['AIC']:.2f}")
    return (best_name, best_predict, best_met), res


# ============================================================
# 8. Статические графики — БЕЗ формулы МНК в заголовке
# ============================================================
def _draw_figure(df, target, best, fixed_axis, fixed_val):
    C0, a0, b0, c0 = fit_power_law(df, target)
    _, best_predict, best_met = best
    fn_f = best_met['formula']

    D_grid = np.linspace(df['D_n'].min(), df['D_n'].max(), 400)

    if fixed_axis == 'p_nom':
        line_vals  = BE_VALS
        palette    = plt.cm.viridis(np.linspace(0.15, 0.85, len(line_vals)))
        cmap       = {v: palette[i] for i, v in enumerate(line_vals)}
        line_label = lambda v: rf'$\bar{{e}}_0$={v}'
        sub_mask   = lambda fv, lv: (df['p_nom']==fv) & (df['bar_e_0']==lv)
        title_top  = rf'$p_{{ном}}$ = {fixed_val/1e6:.0f} МПа'
    else:
        line_vals  = P_VALS
        palette    = plt.cm.plasma(np.linspace(0.1, 0.85, len(line_vals)))
        cmap       = {v: palette[i] for i, v in enumerate(line_vals)}
        line_label = lambda v: f'$p_{{ном}}$={v/1e6:.0f} МПа'
        sub_mask   = lambda fv, lv: (df['bar_e_0']==fv) & (df['p_nom']==lv)
        title_top  = rf'$\bar{{e}}_0$ = {fixed_val}'

    fig, ax = plt.subplots(figsize=(10.5, 7.0))
    fig.subplots_adjust(top=0.82, bottom=0.12, left=0.10, right=0.97)

    for lv in line_vals:
        col = cmap[lv]
        if fixed_axis == 'p_nom':
            p_val, be_val = fixed_val, lv
        else:
            p_val, be_val = lv, fixed_val

        y_mnk = power_law(C0, a0, b0, c0, D_grid,
                          np.full_like(D_grid, p_val, dtype=float),
                          np.full_like(D_grid, be_val, dtype=float))
        ax.plot(D_grid, y_mnk, '-', color=col, lw=2.4, alpha=0.95,
                label=f'{line_label(lv)} — МНК')

        y_fn = best_predict(D_grid,
                            np.full_like(D_grid, p_val, dtype=float),
                            np.full_like(D_grid, be_val, dtype=float))
        ax.plot(D_grid, y_fn, '--', color=col, lw=2.0, alpha=0.85,
                label=f'{line_label(lv)} — {FN_LABEL}')

        sub = df[sub_mask(fixed_val, lv)].sort_values('D_n')
        ax.scatter(sub['D_n'], sub[target], s=85, facecolors='none',
                   edgecolors=col, linewidths=2.2, zorder=5,
                   label=f'{line_label(lv)} — данные')

    ax.set_xlabel('$D_n$, м', fontsize=16)
    ax.set_ylabel(LABELS[target], fontsize=16)
    ax.tick_params(labelsize=15)
    ax.grid(alpha=0.25, linestyle='--')
    ax.legend(fontsize=16, loc='upper right', ncol=1,
              framealpha=0.9, edgecolor='0.7')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

    # Верхняя строка: параметр
    fig.text(0.5, 0.97, title_top, ha='center', va='top', fontsize=16)
    # Только формула функциональной зависимости (без МНК)
    fig.text(0.5, 0.90, 'Функциональная зависимость:', ha='center',
             va='top', fontsize=16)
    fig.text(0.5, 0.87, fn_f, ha='center', va='top', fontsize=16)

    return fig


def plot_static_all(df, target, best, out_dir):
    for p_val in P_VALS:
        fig = _draw_figure(df, target, best, 'p_nom', p_val)
        fn = f'{out_dir}/static_{target}_p_nom_{int(p_val)}_var_bar_e_0.png'
        fig.savefig(fn, dpi=200, bbox_inches='tight'); plt.close(fig)
        print(f'PNG: {fn}')

    for be_val in BE_VALS:
        fig = _draw_figure(df, target, best, 'bar_e_0', be_val)
        be_str = f'{be_val}'.replace('.', '_')
        fn = f'{out_dir}/static_{target}_bar_e_0_{be_str}_var_p_nom.png'
        fig.savefig(fn, dpi=200, bbox_inches='tight'); plt.close(fig)
        print(f'PNG: {fn}')


# ============================================================
# 9. GIF — БЕЗ формулы МНК в заголовке
# ============================================================
def animate_all(df, target, best, out_gif, frames='p_nom'):
    C0, a0, b0, c0 = fit_power_law(df, target)
    _, best_predict, best_met = best
    fn_f = best_met['formula']

    if frames == 'p_nom':
        frame_vals = P_VALS
        line_vals  = BE_VALS
        palette    = plt.cm.viridis(np.linspace(0.15, 0.85, len(line_vals)))
        line_label = lambda v: rf'$\bar{{e}}_0$={v}'
        title_frame = lambda v: rf'$p_{{ном}}$ = {v/1e6:,.0f} МПа'
        mask = lambda v: (df[GROUP_PNOM] == v)
        grp_col = GROUP_BAR
    else:
        frame_vals = BE_VALS
        line_vals  = P_VALS
        palette    = plt.cm.plasma(np.linspace(0.1, 0.85, len(line_vals)))
        line_label = lambda v: f'$p_{{ном}}$={v/1e6:.0f} МПа'
        title_frame = lambda v: rf'$\bar{{e}}_0$ = {v}'
        mask = lambda v: (df[GROUP_BAR] == v)
        grp_col = GROUP_PNOM

    cmap = {v: palette[i] for i, v in enumerate(line_vals)}

    x_min, x_max = df[X_COL].min(), df[X_COL].max()
    y_min, y_max = df[target].min(), df[target].max()
    x_pad = 0.06 * (x_max - x_min)
    y_pad = 0.12 * (y_max - y_min)
    D_grid = np.linspace(x_min, x_max, 400)

    fig, ax = plt.subplots(figsize=(11.5, 7.6))
    fig.subplots_adjust(top=0.80, bottom=0.11, left=0.09, right=0.97)

    title_main = fig.text(0.5, 0.97, '', ha='center', va='top', fontsize=16)
    fig.text(0.5, 0.90, 'Функциональная зависимость:', ha='center',
             va='top', fontsize=16)
    fig.text(0.5, 0.87, fn_f, ha='center', va='top', fontsize=16)

    def update(frame):
        ax.clear()
        fr_val = frame_vals[frame]
        sub = df[mask(fr_val)]

        for lv in line_vals:
            col = cmap[lv]
            if frames == 'p_nom':
                p_val, be_val = fr_val, lv
            else:
                p_val, be_val = lv, fr_val

            y_mnk = power_law(C0, a0, b0, c0, D_grid,
                              np.full_like(D_grid, p_val, dtype=float),
                              np.full_like(D_grid, be_val, dtype=float))
            ax.plot(D_grid, y_mnk, '-', color=col, lw=2.4, alpha=0.95,
                    label=f'{line_label(lv)} — МНК')

            y_fn = best_predict(D_grid,
                                np.full_like(D_grid, p_val, dtype=float),
                                np.full_like(D_grid, be_val, dtype=float))
            ax.plot(D_grid, y_fn, '--', color=col, lw=2.0, alpha=0.85,
                    label=f'{line_label(lv)} — {FN_LABEL}')

            g = sub[sub[grp_col] == lv].sort_values(X_COL)
            if len(g):
                ax.scatter(g[X_COL], g[target], s=85, facecolors='none',
                           edgecolors=col, linewidths=2.2, zorder=5,
                           label=f'{line_label(lv)} — данные')

        ax.set_xlim(x_min - x_pad, x_max + x_pad)
        ax.set_ylim(y_min - y_pad, y_max + y_pad)
        ax.set_xlabel('$D_n$, м', fontsize=16)
        ax.set_ylabel(LABELS[target], fontsize=16)
        ax.tick_params(labelsize=15)
        ax.grid(alpha=0.25, linestyle='--')
        ax.legend(fontsize=16, loc='upper right', ncol=1,
                  framealpha=0.9, edgecolor='0.7')
        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)
        title_main.set_text(title_frame(fr_val))

    ani = animation.FuncAnimation(fig, update,
                                  frames=len(frame_vals),
                                  interval=1500, repeat=True)
    ani.save(out_gif, writer='pillow', fps=1, dpi=110)
    plt.close(fig)
    print(f'GIF: {out_gif}')


# ============================================================
# 10. Запуск
# ============================================================
all_best = {}

for tgt in TARGETS:
    best, table = compare_all_models(df_all, tgt)
    all_best[tgt] = best
    plot_static_all(df_all, tgt, best, OUT_DIR)
    animate_all(df_all, tgt, best,
                f'{OUT_DIR}/anim_{tgt}_frames_p_nom.gif', frames='p_nom')
    animate_all(df_all, tgt, best,
                f'{OUT_DIR}/anim_{tgt}_frames_bar_e_0.gif', frames='bar_e_0')

print('\nГотово. Файлы в:', os.path.abspath(OUT_DIR))

summary = [(t, n, m['R2'], m['SKO'], m['AIC'], m['k'])
           for t, (n, _, m) in all_best.items()]
print("\n" + "="*90)
print("ИТОГ: лучшие модели по AIC")
print("="*90)
print(pd.DataFrame(summary,
                   columns=['Цель', 'Модель', 'R²', 'СКО', 'AIC', 'k']
                   ).round(6).to_string(index=False))

Всего строк: 63 | валидных: 63

СРАВНЕНИЕ МОДЕЛЕЙ: delta_I_p  (сортировка по AIC)
                      Модель       R²  САО (MAE)  СКО (RMSE)         AIC  k
H5: степень·exp(k·D+m·ē+n·p) 0.997033   0.005965    0.007728 -598.723887  7
     H7: сумма двух степеней 0.996961   0.005857    0.007821 -595.218247  8
    H6: степень + A·exp(k·D) 0.995695   0.007043    0.009309 -577.273716  6
    H4: степень·exp(k·D+m·ē) 0.995652   0.007016    0.009355 -576.650945  6
       H8: D^(a+q·ē)·p^b·ē^c 0.995046   0.007115    0.009985 -570.436436  5
        H2: степень·exp(k·D) 0.994980   0.007317    0.010052 -569.598032  5
       H9: D^(a+r·p)·p^b·ē^c 0.994924   0.007713    0.010108 -568.903217  5
        H3: степень·exp(m·ē) 0.994850   0.007455    0.010181 -567.987708  5
      H1: степень (МНК-ядро) 0.994182   0.007874    0.010822 -562.297232  4

>>> ЛУЧШАЯ ПО AIC: H5: степень·exp(k·D+m·ē+n·p)
    R²=0.997033, СКО=0.007728, AIC=-598.72
PNG: results/static_delta_I_p_p_nom_4000000_var_bar_e_0.png
PNG: r